# JSB Chorales: Preprocessing & Tokenization

This notebook builds on the parsed `(pitch, duration)` voice sequences from step 1. We construct a token vocabulary, encode sequences as integers, split chorales 80/10/10, and create PyTorch `Dataset` / `DataLoader` objects using a sliding window of 32 tokens.

## Approach

- **Token**: a `(pitch, duration)` tuple; rests use `None` for pitch.
- **Vocabulary**: built from **training chorales only** (reserved `PAD` and `UNK` tokens).
- **Unit of data**: each SATB voice is treated as its own sequence (4 sequences per chorale).
- **Split**: 80/10/10 at the **chorale** level to avoid leakage across train/val/test.
- **Windowing**: sliding windows of size 32 with stride 1; each sample is `(x, y)` where `y` is the next-token target shifted by one position (both length 32).

In [1]:
from collections import Counter

import torch
from torch.utils.data import DataLoader

from chorale_data import (
    PAD_TOKEN,
    UNK_TOKEN,
    PitchDurationVocab,
    SlidingWindowDataset,
    build_dataloaders,
    flatten_voice_sequences,
    load_chorales,
    split_chorale_indices,
)

WINDOW_SIZE = 32
BATCH_SIZE = 64
SEED = 42

## Load parsed chorales

In [2]:
encoded_chorales, metadata = load_chorales()
num_chorales = len(encoded_chorales)

print(f"Loaded {num_chorales} four-voice chorales.")
print(f"Example raw sequence (Soprano, first 8 tokens):")
print(encoded_chorales[0][0][:8])

Loaded 351 four-voice chorales.
Example raw sequence (Soprano, first 8 tokens):
[(67, 1.0), (67, 2.0), (74, 1.0), (71, 1.5), (69, 0.5), (67, 1.0), (67, 1.5), (69, 0.5)]


## Chorale-level train / val / test split

In [3]:
splits = split_chorale_indices(num_chorales, train_ratio=0.8, val_ratio=0.1, seed=SEED)

train_sequences = flatten_voice_sequences(encoded_chorales, splits.train_indices)
val_sequences = flatten_voice_sequences(encoded_chorales, splits.val_indices)
test_sequences = flatten_voice_sequences(encoded_chorales, splits.test_indices)

print(f"Chorales — train: {len(splits.train_indices)}, val: {len(splits.val_indices)}, test: {len(splits.test_indices)}")
print(f"Voice sequences — train: {len(train_sequences)}, val: {len(val_sequences)}, test: {len(test_sequences)}")

Chorales — train: 280, val: 35, test: 36
Voice sequences — train: 1120, val: 140, test: 144


## Build vocabulary from training tokens

In [4]:
vocab = PitchDurationVocab()
vocab.build_from_sequences(train_sequences)

print(f"Vocabulary size (including PAD/UNK): {len(vocab)}")
print(f"Special tokens: PAD={vocab.token_to_id[PAD_TOKEN]}, UNK={vocab.token_to_id[UNK_TOKEN]}")

token_counts = Counter(token for seq in train_sequences for token in seq)
most_common = token_counts.most_common(10)
print("\nTop 10 training tokens:")
for token, count in most_common:
    pitch, duration = token
    pitch_label = "REST" if pitch is None else f"MIDI {pitch}"
    print(f"  id={vocab.token_id(token):4d}  ({pitch_label}, dur={duration})  count={count}")

Vocabulary size (including PAD/UNK): 323
Special tokens: PAD=0, UNK=1

Top 10 training tokens:
  id=  14  (MIDI 62, dur=1.0)  count=2726
  id=   2  (MIDI 67, dur=1.0)  count=2617
  id=   3  (MIDI 69, dur=1.0)  count=2406
  id=  15  (MIDI 64, dur=1.0)  count=2267
  id=  35  (MIDI 57, dur=1.0)  count=1941
  id=  22  (MIDI 62, dur=0.5)  count=1755
  id=  16  (MIDI 65, dur=1.0)  count=1640
  id=  39  (MIDI 57, dur=0.5)  count=1582
  id=  31  (MIDI 60, dur=1.0)  count=1507
  id=  30  (MIDI 64, dur=0.5)  count=1464


## Encode all sequences as integers

In [5]:
encoded_train = [vocab.encode(seq) for seq in train_sequences]
encoded_val = [vocab.encode(seq) for seq in val_sequences]
encoded_test = [vocab.encode(seq) for seq in test_sequences]

unk_id = vocab.token_to_id[UNK_TOKEN]
unk_counts = {
    "train": sum(ids.count(unk_id) for ids in encoded_train),
    "val": sum(ids.count(unk_id) for ids in encoded_val),
    "test": sum(ids.count(unk_id) for ids in encoded_test),
}

print("Encoded sequence counts:")
print(f"  train tokens: {sum(len(s) for s in encoded_train):,}")
print(f"  val tokens:   {sum(len(s) for s in encoded_val):,}")
print(f"  test tokens:  {sum(len(s) for s in encoded_test):,}")
print(f"\nUNK token usage (should be 0 outside train if vocab is complete): {unk_counts}")

sample_raw = train_sequences[0][:8]
sample_ids = encoded_train[0][:8]
print("\nSample encoding (first 8 tokens of first train voice):")
for raw, token_id in zip(sample_raw, sample_ids):
    print(f"  {raw} -> {token_id}")

Encoded sequence counts:
  train tokens: 66,863
  val tokens:   7,652
  test tokens:  7,531

UNK token usage (should be 0 outside train if vocab is complete): {'train': 0, 'val': 0, 'test': 6}

Sample encoding (first 8 tokens of first train voice):
  (67, 1.0) -> 2
  (67, 1.0) -> 2
  (69, 1.0) -> 3
  (67, 1.0) -> 2
  (69, 0.5) -> 4
  (71, 0.5) -> 5
  (72, 1.0) -> 6
  (72, 1.0) -> 6


## Sliding-window Dataset and DataLoaders

In [6]:
train_dataset = SlidingWindowDataset(encoded_train, window_size=WINDOW_SIZE)
val_dataset = SlidingWindowDataset(encoded_val, window_size=WINDOW_SIZE)
test_dataset = SlidingWindowDataset(encoded_test, window_size=WINDOW_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Window size: {WINDOW_SIZE}")
print(f"Training windows:   {len(train_dataset):,}")
print(f"Validation windows:   {len(val_dataset):,}")
print(f"Test windows:         {len(test_dataset):,}")
print(f"Batches per epoch:    {len(train_loader)} (batch_size={BATCH_SIZE})")

Window size: 32
Training windows:   31,113
Validation windows:   3,172
Test windows:         2,925
Batches per epoch:    487 (batch_size=64)


In [7]:
x_batch, y_batch = next(iter(train_loader))

print(f"Input batch shape:  {tuple(x_batch.shape)}  (batch, window)")
print(f"Target batch shape: {tuple(y_batch.shape)}  (batch, window)")
print(f"Input dtype: {x_batch.dtype}, Target dtype: {y_batch.dtype}")

print("\nFirst training example:")
x0, y0 = train_dataset[0]
print(f"  x ids: {x0.tolist()}")
print(f"  y ids: {y0.tolist()}")
print(f"  x tokens: {vocab.decode(x0.tolist())}")
print(f"  y tokens: {vocab.decode(y0.tolist())}")

Input batch shape:  (64, 32)  (batch, window)
Target batch shape: (64, 32)  (batch, window)
Input dtype: torch.int64, Target dtype: torch.int64

First training example:
  x ids: [2, 2, 3, 2, 4, 5, 6, 6, 7, 8, 9, 10, 8, 5, 11, 9, 12, 9, 9, 6, 5, 4, 12, 13, 2, 2, 3, 2, 4, 5, 6, 6]
  y ids: [2, 3, 2, 4, 5, 6, 6, 7, 8, 9, 10, 8, 5, 11, 9, 12, 9, 9, 6, 5, 4, 12, 13, 2, 2, 3, 2, 4, 5, 6, 6, 7]
  x tokens: [(67, 1.0), (67, 1.0), (69, 1.0), (67, 1.0), (69, 0.5), (71, 0.5), (72, 1.0), (72, 1.0), (71, 1.0), (72, 2.0), (74, 1.0), (76, 1.0), (72, 2.0), (71, 0.5), (72, 0.5), (74, 1.0), (69, 2.0), (74, 1.0), (74, 1.0), (72, 1.0), (71, 0.5), (69, 0.5), (69, 2.0), (67, 2.0), (67, 1.0), (67, 1.0), (69, 1.0), (67, 1.0), (69, 0.5), (71, 0.5), (72, 1.0), (72, 1.0)]
  y tokens: [(67, 1.0), (69, 1.0), (67, 1.0), (69, 0.5), (71, 0.5), (72, 1.0), (72, 1.0), (71, 1.0), (72, 2.0), (74, 1.0), (76, 1.0), (72, 2.0), (71, 0.5), (72, 0.5), (74, 1.0), (69, 2.0), (74, 1.0), (74, 1.0), (72, 1.0), (71, 0.5), (69, 0.5), 

In [8]:
# Convenience wrapper that rebuilds loaders from splits + vocab
train_loader2, val_loader2, test_loader2 = build_dataloaders(
    encoded_chorales,
    splits,
    vocab,
    window_size=WINDOW_SIZE,
    batch_size=BATCH_SIZE,
)

assert len(train_loader2) == len(train_loader)
assert len(val_loader2) == len(val_loader)
assert len(test_loader2) == len(test_loader)
print("build_dataloaders() helper produces matching loader sizes.")

build_dataloaders() helper produces matching loader sizes.


## Summary

- Built a `(pitch, duration)` vocabulary from training voices only.
- Encoded every voice sequence as a list of integer token ids.
- Split **351 chorales** into 80/10/10 train/val/test (then expanded to **4 voice sequences per chorale**).
- Created `SlidingWindowDataset` examples of length 32 with next-token targets for autoregressive modeling.
- Wrapped datasets in PyTorch `DataLoader`s ready for model training.